# 🔬 Entrenamiento de YOLOv8 para Detección de Equipos de Bromatología - UTEQ
### Proyecto: Asistente Móvil Inteligente con Visión Artificial y RAG
**Universidad Técnica Estatal de Quevedo (UTEQ)**  
**Facultad de Ciencias Pecuarias y Biológicas - Laboratorio de Bromatología**

---

Este cuaderno permite entrenar un modelo **YOLOv8** personalizado para detectar **20 equipos, instrumentos y materiales** del laboratorio, evaluarlo y exportarlo automáticamente a formato **TensorFlow Lite (`.tflite`)** para la aplicación móvil Android.

## 🛠️ Paso 1: Configurar GPU y Dependencias
Asegúrate de que en el menú superior esté activado el entorno de ejecución con **GPU** (`Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4`).

In [ ]:
# Verificar aceleración por GPU
!nvidia-smi

# Instalar Ultralytics YOLO y dependencias necesarias
!pip install -q ultralytics tensorflow albumentations opencv-python matplotlib

import ultralytics
ultralytics.checks()

## 📦 Paso 2: Cargar y Descomprimir el Dataset de Bromatología
Sube el archivo `dataset_bromatologia_uteq.zip` generado desde tu proyecto local o descárgalo aquí.

In [ ]:
import os
import zipfile
from google.colab import files

# Si el archivo no existe en el entorno, solicitar la subida del .zip
if not os.path.exists('dataset_bromatologia_uteq.zip'):
    print("Por favor selecciona el archivo 'dataset_bromatologia_uteq.zip' de tu computadora:")
    uploaded = files.upload()

# Descomprimir el dataset
print("Descomprimiendo dataset...")
with zipfile.ZipFile('dataset_bromatologia_uteq.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

print("✅ Dataset descomprimido exitosamente.")
!ls -lh dataset

## ⚙️ Paso 3: Verificar `data.yaml` con las 20 Clases

In [ ]:
# Ajustar data.yaml con rutas absolutas de Colab
data_yaml_content = """
path: /content/dataset
train: images/train
val: images/val
test: images/test

nc: 20
names:
  0: destilador_kjeldahl
  1: analizador_fibra
  2: placa_calefactora_heidolph
  3: phmetro_ohaus
  4: molino_ciclonico_foss
  5: estufa_secado_memmert
  6: refractometro_atago
  7: calorimetro_bomba
  8: campana_extraccion_gases
  9: cabina_flujo_laminar_uvp
  10: sistema_tratamiento_agua
  11: destilador_agua
  12: bomba_vacio_recirculacion
  13: bomba_vacio_membrana
  14: agitador_vortex
  15: gradilla_tubos_kjeldahl
  16: gradilla_pipetas
  17: piseta_reactivo
  18: cilindro_gas
  19: bidon_agua_destilada
"""

with open('data.yaml', 'w') as f:
    f.write(data_yaml_content.strip())

print("✅ Archivo data.yaml configurado para Colab:")
!cat data.yaml

## 🚀 Paso 4: Entrenar el Modelo YOLOv8 en GPU
Utilizaremos la arquitectura **YOLOv8 Nano (`yolov8n.pt`)** optimizada para inferencia en tiempo real en dispositivos móviles Android.

In [ ]:
from ultralytics import YOLO

# Cargar modelo base preentrenado en COCO
model = YOLO('yolov8n.pt')

# Iniciar entrenamiento
results = model.train(
    data='data.yaml',
    epochs=50,          # 50 épocas con early stopping
    imgsz=640,          # Tamaño de entrada estándar para TFLite
    batch=16,           # Tamaño de lote optimizado para GPU T4
    patience=15,        # Parar si no mejora en 15 épocas
    save=True,
    device=0,           # Usar GPU 0
    workers=4,
    project='yolo_uteq_bromatologia',
    name='train_v1',
    exist_ok=True,
    # Aumentación de datos para laboratorio
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    scale=0.5
)

print("🎉 ¡Entrenamiento completado exitosamente!")

## 📊 Paso 5: Evaluar Rendimiento y Métricas (mAP & Matriz de Confusión)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Evaluar en conjunto de validación
metrics = model.val()
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")

# Mostrar matriz de confusión y curvas de aprendizaje
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

cf_path = 'yolo_uteq_bromatologia/train_v1/confusion_matrix.png'
res_path = 'yolo_uteq_bromatologia/train_v1/results.png'

if os.path.exists(cf_path):
    axes[0].imshow(mpimg.imread(cf_path))
    axes[0].set_title('Matriz de Confusión')
    axes[0].axis('off')

if os.path.exists(res_path):
    axes[1].imshow(mpimg.imread(res_path))
    axes[1].set_title('Curvas de Pérdida y Métricas')
    axes[1].axis('off')

plt.tight_layout()
plt.show()

## 📱 Paso 6: Exportar Modelo a TensorFlow Lite (`.tflite`)

In [ ]:
import shutil

# Cargar los mejores pesos obtenidos
best_model = YOLO('yolo_uteq_bromatologia/train_v1/weights/best.pt')

# Exportar a TensorFlow Lite con tamaño 640x640
exported_path = best_model.export(format='tflite', imgsz=640)
print(f"Modelo exportado a: {exported_path}")

# Renombrar y copiar para la aplicación Android
tflite_src = 'yolo_uteq_bromatologia/train_v1/weights/best_saved_model/best_float32.tflite'
if not os.path.exists(tflite_src):
    tflite_src = 'yolo_uteq_bromatologia/train_v1/weights/best_float32.tflite'
if not os.path.exists(tflite_src):
    tflite_src = 'yolo_uteq_bromatologia/train_v1/weights/best.tflite'

target_tflite = 'yolov8_bromatologia.tflite'
shutil.copyfile(tflite_src, target_tflite)
print(f"✅ Archivo final para Android listo: {target_tflite} ({os.path.getsize(target_tflite)/(1024*1024):.2f} MB)")

## 🧪 Paso 7: Probar Inferencia en una Foto del Test Set

In [ ]:
import glob

test_images = glob.glob('dataset/images/test/*.jpg') + glob.glob('dataset/images/test/*.png')
if test_images:
    sample_img = test_images[0]
    print(f"Probando inferencia en: {sample_img}")
    
    # Predecir con el modelo
    pred_results = best_model.predict(source=sample_img, conf=0.35, save=True, project='test_results', name='demo')
    
    # Mostrar resultado con bounding boxes
    result_img = glob.glob('test_results/demo/*.jpg')[0]
    plt.figure(figsize=(10, 8))
    plt.imshow(mpimg.imread(result_img))
    plt.title('Detección YOLOv8 Bromatología UTEQ')
    plt.axis('off')
    plt.show()

## ⬇️ Paso 8: Descargar el Archivo `.tflite` para la App Android

In [ ]:
from google.colab import files

print("Descargando yolov8_bromatologia.tflite a tu computadora...")
files.download('yolov8_bromatologia.tflite')

# Guardar también labels.txt
labels_text = """destilador_kjeldahl
analizador_fibra
placa_calefactora_heidolph
phmetro_ohaus
molino_ciclonico_foss
estufa_secado_memmert
refractometro_atago
calorimetro_bomba
campana_extraccion_gases
cabina_flujo_laminar_uvp
sistema_tratamiento_agua
destilador_agua
bomba_vacio_recirculacion
bomba_vacio_membrana
agitador_vortex
gradilla_tubos_kjeldahl
gradilla_pipetas
piseta_reactivo
cilindro_gas
bidon_agua_destilada"""

with open('labels.txt', 'w') as f:
    f.write(labels_text.strip())

files.download('labels.txt')
print("✅ Coloca ambos archivos en la carpeta: app/src/main/assets/ de tu proyecto Android.")